# QEC QCNN decoder — Kaggle GPU training

Trains the QCNN decoders (Cong + Hybrid QCCNN) and a classical CNN reference on
a GPU using PennyLane's Torch-native `default.qubit` statevector, which runs on
CUDA and backprops the whole minibatch at once, then evaluates all of them
against MWPM and plots the comparison.

**Before running:**
1. Notebook settings -> **Accelerator: GPU** (T4 or P100).
2. Notebook settings -> **Internet: On** (needed to clone the repo + pip install).

CPU tests and the HPC path are unaffected — this only sets
`QEC_QML_DEVICE=default.qubit` and `--device cuda`.

In [ ]:
# 1. Clone the repo (public, default branch = main)
%cd /kaggle/working
!rm -rf Quantum-Error-Correction-Decoder
!git clone --depth 1 \
    https://github.com/TuanKiet16/Quantum-Error-Correction-Decoder.git
%cd Quantum-Error-Correction-Decoder

In [ ]:
# 2. Install. Torch is preinstalled on Kaggle GPU images; install the rest.
!pip install -q stim pymatching pennylane 'pennylane-lightning' loguru
# The package itself (pulls nothing heavy since torch already present):
!pip install -q -e . --no-deps

In [ ]:
# 3. Confirm the GPU is visible
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'Enable GPU accelerator in notebook settings'

In [ ]:
# 4. GPU backend for PennyLane, for the whole process
import os
os.environ['QEC_QML_DEVICE'] = 'default.qubit'

In [ ]:
# 5. Quick smoke run to prove the GPU path works (seconds)
!QEC_QML_DEVICE=default.qubit python -m qec_decoder.train \
    --model qcnn_cong --d 3 --ps 0.005 --shots 400 --epochs 1 \
    --batch-size 128 --device cuda

## Choosing distances d = 3, 5, 7 (and the budget)

Edit **`DISTANCES`** in the next cell. Cost scales steeply because QCNN-Cong runs
one circuit *per patch*, and patches grow with d (d3≈2, d5≈10, d7≈28).

**Kaggle limits:** 30h is the *weekly* GPU quota; a *single* Save & Run All
(commit) run is capped ~12h. A commit that hits the wall **fails and may lose its
output**, so don't pack more than ~12h into one commit.

**Recommended way to get all of 3, 5, 7:**
- **Commit 1:** `DISTANCES = [3, 5]`  (~4–6h)
- **Commit 2:** `DISTANCES = [7]`     (d7 cong is the heavy one)

Each `(model, d)` checkpoint is written the moment it finishes, and the loop
**skips any checkpoint that already exists**, so re-running a commit continues
instead of redoing finished work.

**To do 3, 5, 7 in one commit anyway:** set `DISTANCES = [3, 5, 7]` *and* cut
cost, e.g. `SHOTS = 10000`, `EPOCHS = 40`, or drop `qcnn_cong` from `MODELS`
(hybrid/cnn are cheap at any d).

**Keep the eval cell (6b) `--ds` in sync** with whatever you trained — distances
without a checkpoint only get the MWPM baseline.

In [ ]:
# 6. Full training. Cost = shots x #p x (patches per sample) x epochs.
#    Edit DISTANCES / SHOTS / EPOCHS / MODELS below; see the markdown above.
import subprocess, itertools, os

MODELS    = ['qcnn_cong', 'qcnn_hybrid', 'cnn']
DISTANCES = [3, 5]           # -> for d7, run a second commit with DISTANCES = [7]
PS        = ['0.003', '0.005', '0.008', '0.01', '0.015']
SHOTS     = 20000            # per p; raise for tighter fits if time allows
EPOCHS    = 60               # enough gradient steps with the new architecture
QCHUNK    = 2048             # lower (e.g. 1024) if you hit OOM; higher = faster

for model, d in itertools.product(MODELS, DISTANCES):
    ckpt = f'checkpoints/{model}_d{d}.pt'
    if os.path.exists(ckpt):
        print('skip (already trained):', ckpt)
        continue
    cmd = ['python', '-m', 'qec_decoder.train',
           '--model', model, '--d', str(d), '--ps', *PS,
           '--shots', str(SHOTS), '--epochs', str(EPOCHS),
           '--batch-size', '256', '--device', 'cuda', '--qchunk', str(QCHUNK)]
    print('>>>', ' '.join(cmd))
    env = {**os.environ, 'QEC_QML_DEVICE': 'default.qubit'}
    subprocess.run(cmd, env=env, check=True)

In [ ]:
# 6b. Evaluate all trained decoders vs MWPM on a fresh test set (GPU).
#     Keep --ds in sync with DISTANCES above (missing-checkpoint distances only
#     get MWPM). Writes results/comparison.json (p_L, uncertainty, epsilon_d,
#     fidelity, per-decoder Lambda). Test seed is disjoint from training.
!QEC_QML_DEVICE=default.qubit python -m qec_decoder.evaluate \
    --ckpt-dir checkpoints --ds 3 5 \
    --ps 0.003 0.005 0.008 0.01 0.015 \
    --shots 20000 --device cuda --out results/comparison.json

In [ ]:
# 6c. Plot the comparison (p_L vs p per distance) and show it inline.
!pip install -q matplotlib pdf2image
!python -m qec_decoder.plot_comparison --out figures/decoder_comparison.pdf
try:
    from pdf2image import convert_from_path
    from IPython.display import display
    display(convert_from_path('figures/decoder_comparison.pdf', dpi=110)[0])
except Exception as e:
    print('plot saved to figures/decoder_comparison.pdf (inline preview skipped:', e, ')')

In [ ]:
# 7. Bundle checkpoints + results + comparison + plot to download from Output tab
!mkdir -p /kaggle/working/out
!cp -r checkpoints results figures /kaggle/working/out/ 2>/dev/null || true
!cd /kaggle/working && zip -qr qec_qcnn_output.zip out && echo 'wrote qec_qcnn_output.zip'
!python -c "import json; d=json.load(open('results/comparison.json')); print('Lambda:', d['lambda'])" 2>/dev/null || true
!ls -la checkpoints figures 2>/dev/null || true